# Anomaly Detection - Burn-in Data

In [1]:
import json
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', None)

## Config

In [2]:
CSV_PATH = "burn_in_dataset.csv"
SPLIT_PATH = "component_split.csv"

## Shared train/val/test split

In [3]:
def build_component_split(df, batch_col='Batch_ID', label_col='Label', seed=134, block=5):
    batch_fail = df.groupby(batch_col)[label_col].apply(lambda s: (s == 'Fail').mean())
    order = batch_fail.sort_values().index.tolist()
    rng = np.random.RandomState(seed)
    cycle_base = ['train', 'train', 'train', 'val', 'test']
    split_map = {}
    for i in range(0, len(order), block):
        chunk = order[i:i + block]
        if len(chunk) == block:
            roles = cycle_base.copy()
        else:
            roles = list(rng.choice(cycle_base, size=len(chunk), replace=False))
        rng.shuffle(roles)
        for b, r in zip(chunk, roles):
            split_map[b] = r
    out = df[['Component_ID', batch_col]].copy()
    out['split'] = out[batch_col].map(split_map)
    return out

raw_df = pd.read_csv(CSV_PATH)
if Path(SPLIT_PATH).exists():
    split_df = pd.read_csv(SPLIT_PATH)
else:
    split_df = build_component_split(raw_df)
    split_df.to_csv(SPLIT_PATH, index=False)

print(split_df['split'].value_counts())
_check = raw_df.merge(split_df[['Component_ID','split']], on='Component_ID')
print(_check.groupby('split')['Label'].value_counts(normalize=True).unstack().round(3))

split
train    5809
test     2236
val      1955
Name: count, dtype: int64
Label  Borderline   Fail   Safe
split                          
test        0.186  0.116  0.699
train       0.210  0.119  0.671
val         0.229  0.129  0.642


## Pipeline config + validation

In [4]:
@dataclass
class PipelineConfig:
    id_col: str = "Component_ID"
    lot_col: str = "Batch_ID"
    group_col: str = "Part_Type"
    absolute_result_col: str = "Traditional_Test_Result"
    absolute_fail_value: str = "Fail"
    label_col: str = "Label"
    params: List[str] = field(default_factory=lambda: ["Leakage"])  # Resistance/Vth columns removed from the dataset
    early_timepoints: List[str] = field(default_factory=lambda: ["0h", "24h"])
    weight_lot_relative: float = 0.40
    weight_multivariate: float = 0.40
    weight_drift: float = 0.20
    threshold_monitor: float = 30.0
    threshold_hold: float = 60.0
    threshold_reject: float = 80.0
    z_score_cap: float = 8.0
    iforest_n_estimators: int = 300
    iforest_contamination: str = "auto"
    iforest_min_group_size: int = 30
    random_state: int = 42

    def __post_init__(self):
        total = self.weight_lot_relative + self.weight_multivariate + self.weight_drift
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Layer weights must sum to 1.0, got {total:.3f}")
        if not (self.threshold_monitor < self.threshold_hold < self.threshold_reject):
            raise ValueError("Thresholds must be strictly increasing")

    def value_col(self, param, timepoint): return f"{param}_{timepoint}"
    def delta_col(self, param): return f"{param}_delta_24h"
    def pct_change_col(self, param): return f"{param}_pct_change_24h"
    def early_value_cols(self):
        return [self.value_col(p, t) for p in self.params for t in self.early_timepoints]
    def lot_relative_cols(self):
        latest = self.early_timepoints[-1]
        return [self.value_col(p, latest) for p in self.params] + [self.delta_col(p) for p in self.params]
    def multivariate_feature_cols(self):
        cols = []
        for p in self.params:
            cols += [self.value_col(p, t) for t in self.early_timepoints]
            cols += [self.delta_col(p), self.pct_change_col(p)]
        return cols
    def required_input_cols(self):
        # NOTE: absolute_result_col (Traditional_Test_Result) is intentionally NOT required here.
        # It's the ground-truth outcome of the full-duration static-limit check, which by definition
        # doesn't exist yet for a component that's still mid burn-in - that's exactly what a caller
        # is asking this pipeline to help assess early. See AbsoluteSpecLayer below for how it's
        # handled when present (backtest/training data) vs absent (live/new components).
        return [self.id_col, self.lot_col, self.group_col] + self.early_value_cols()


class SchemaError(ValueError): pass

def validate_schema(df, config, require_label=False):
    required = list(config.required_input_cols())
    if require_label: required.append(config.label_col)
    missing = [c for c in required if c not in df.columns]
    if missing: raise SchemaError(f"Input data is missing required columns: {missing}")

## Imputation, drift features, the three scoring layers

In [5]:
class LotMedianImputer:
    def __init__(self, config):
        self.config = config; self.columns = config.early_value_cols()
        self.group_medians_ = {}; self.global_medians_ = {}
    def fit(self, df):
        lot_col = self.config.lot_col
        for col in self.columns:
            self.group_medians_[col] = df.groupby(lot_col)[col].median().to_dict()
            self.global_medians_[col] = float(df[col].median())
        return self
    def transform(self, df):
        df = df.copy(); lot_col = self.config.lot_col
        for col in self.columns:
            if df[col].isna().sum() == 0: continue
            fallback = df[lot_col].map(self.group_medians_[col]).fillna(self.global_medians_[col])
            df[col] = df[col].fillna(fallback)
        return df
    def fit_transform(self, df): return self.fit(df).transform(df)


def add_drift_features(df, config):
    df = df.copy(); first_tp, last_tp = config.early_timepoints[0], config.early_timepoints[-1]
    for p in config.params:
        v_first = df[config.value_col(p, first_tp)]; v_last = df[config.value_col(p, last_tp)]
        df[config.delta_col(p)] = v_last - v_first
        df[config.pct_change_col(p)] = np.where(v_first != 0, (v_last - v_first) / v_first.abs(), 0.0)
    return df


class AbsoluteSpecLayer:
    """Static datasheet-limit check.
    - When Traditional_Test_Result is present in the input (historical/backtest data, where the
      full burn-in already ran) we use that ground truth directly - unchanged behaviour.
    - When it's absent (a genuinely new component, still only at 0h/24h) there is no way to know
      the *final* static-limit verdict yet, so instead we flag anything that has ALREADY breached
      the absolute datasheet limit on the checkpoints we do have. Whether it clears the check later
      is exactly what Module B's drift prediction + this layer's live re-check (once 96h/168h data
      arrives) are for. The limits themselves are learned once from the full historical dataset at
      fit() time (its all-checkpoint max per parameter), so no extra file/config is needed.
    """
    def __init__(self, config): self.config = config; self.limits_ = {}
    def fit(self, df):
        for p in self.config.params:
            cols = [c for c in df.columns if c.startswith(f'{p}_') and c[len(p)+1:].endswith('h')]
            if cols:
                self.limits_[p] = float(df[cols].max().max())
        return self
    def transform(self, df):
        col, fail_value = self.config.absolute_result_col, self.config.absolute_fail_value
        if col in df.columns:
            return (df[col].astype(str).str.strip().str.lower() == fail_value.lower()).astype(int)
        exceeded = pd.Series(False, index=df.index)
        for p in self.config.params:
            limit = self.limits_.get(p)
            if limit is None:
                continue
            for t in self.config.early_timepoints:
                c = self.config.value_col(p, t)
                if c in df.columns:
                    exceeded |= (df[c] >= limit)
        return exceeded.astype(int)


def _robust_median_mad(series):
    median = float(series.median()); mad = float((series - median).abs().median()) * 1.4826
    return median, mad

class LotRelativeScorer:
    def __init__(self, config):
        self.config = config; self.columns = config.lot_relative_cols()
        self.group_stats_ = {}; self.global_stats_ = {}
    def fit(self, df):
        lot_col = self.config.lot_col
        for col in self.columns:
            self.group_stats_[col] = df.groupby(lot_col)[col].apply(_robust_median_mad).to_dict()
            self.global_stats_[col] = _robust_median_mad(df[col])
        return self
    def transform(self, df):
        lot_col = self.config.lot_col
        abs_z_frame = pd.DataFrame(index=df.index)
        for col in self.columns:
            medians = df[lot_col].map(lambda lot: self.group_stats_[col].get(lot, (np.nan, np.nan))[0])
            mads = df[lot_col].map(lambda lot: self.group_stats_[col].get(lot, (np.nan, np.nan))[1])
            g_med, g_mad = self.global_stats_[col]
            medians = medians.fillna(g_med)
            mads = mads.fillna(g_mad).replace(0, g_mad if g_mad > 0 else 1e-6)
            abs_z_frame[col] = ((df[col] - medians) / mads).abs()
        worst_z = abs_z_frame.max(axis=1)
        score = (worst_z.clip(upper=self.config.z_score_cap) / self.config.z_score_cap) * 100
        return score, worst_z


GLOBAL_KEY = "__global__"
class MultivariateAnomalyScorer:
    def __init__(self, config):
        self.config = config; self.feature_cols = config.multivariate_feature_cols()
        self.models_ = {}; self.score_bounds_ = {}
        self.background_ = {}   # FIX: per-group background sample, stored at fit time, for SHAP later

    def _fit_one(self, X):
        pipeline = Pipeline([("scaler", StandardScaler()),
                              ("iforest", IsolationForest(n_estimators=self.config.iforest_n_estimators,
                                                           contamination=self.config.iforest_contamination,
                                                           random_state=self.config.random_state))])
        pipeline.fit(X)
        raw = -pipeline.decision_function(X)
        p1, p99 = np.percentile(raw, [1, 99])
        return pipeline, (float(p1), float(p99))

    def fit(self, df):
        group_col = self.config.group_col
        n_bg = min(100, len(df))
        model, bounds = self._fit_one(df[self.feature_cols].values)
        self.models_[GLOBAL_KEY] = model; self.score_bounds_[GLOBAL_KEY] = bounds
        self.background_[GLOBAL_KEY] = df[self.feature_cols].sample(n_bg, random_state=self.config.random_state)
        for group_value, group_df in df.groupby(group_col):
            if len(group_df) < self.config.iforest_min_group_size:
                print(f"WARNING: group '{group_value}' has only {len(group_df)} rows; using global fallback")
                continue
            model, bounds = self._fit_one(group_df[self.feature_cols].values)
            self.models_[str(group_value)] = model; self.score_bounds_[str(group_value)] = bounds
            n_bg_g = min(100, len(group_df))
            self.background_[str(group_value)] = group_df[self.feature_cols].sample(n_bg_g, random_state=self.config.random_state)
        return self

    def transform(self, df):
        group_col = self.config.group_col
        scores = pd.Series(index=df.index, dtype=float)
        for group_value, group_df in df.groupby(group_col):
            key = str(group_value) if str(group_value) in self.models_ else GLOBAL_KEY
            model = self.models_[key]; p1, p99 = self.score_bounds_[key]
            raw = -model.decision_function(group_df[self.feature_cols].values)
            norm = np.clip((raw - p1) / (p99 - p1 + 1e-9), 0, 1) * 100
            scores.loc[group_df.index] = norm
        return scores

    def shap_top_features(self, df, top_k=3):
        group_col = self.config.group_col
        out = pd.Series(index=df.index, dtype=object)
        for group_value, group_df in df.groupby(group_col):
            key = str(group_value) if str(group_value) in self.models_ else GLOBAL_KEY
            model = self.models_[key]
            background = self.background_[key]
            score_fn = lambda X, _m=model: -_m.decision_function(X)
            explainer = shap.Explainer(score_fn, background.values, feature_names=self.feature_cols)
            sv = explainer(group_df[self.feature_cols].values)
            for i, idx in enumerate(group_df.index):
                vals = np.asarray(sv.values[i]).reshape(-1)
                order = np.argsort(-np.abs(vals))[:top_k]
                out.loc[idx] = ', '.join(f'{self.feature_cols[j]}({vals[j]:+.2f})' for j in order)
        return out


class DriftSeverityScorer:
    def __init__(self, config): self.config = config; self.bound_ = None
    def _severity(self, df):
        cols = [self.config.pct_change_col(p) for p in self.config.params]
        return df[cols].abs().max(axis=1)
    def fit(self, df):
        bound = float(self._severity(df).quantile(0.99)); self.bound_ = bound if bound > 0 else 1e-6
        return self
    def transform(self, df):
        return (self._severity(df).clip(upper=self.bound_) / self.bound_) * 100


def combine(lot_score, mv_score, drift_score, absolute_fail, config):
    combined = (config.weight_lot_relative*lot_score + config.weight_multivariate*mv_score + config.weight_drift*drift_score)
    combined = np.where(absolute_fail == 1, np.maximum(combined, config.threshold_reject + 5), combined)
    return pd.Series(np.round(np.clip(combined, 0, 100), 1), index=lot_score.index)


def decide(risk_score, config):
    bins = [-np.inf, config.threshold_monitor, config.threshold_hold, config.threshold_reject, np.inf]
    return pd.cut(risk_score, bins=bins, labels=["PASS","MONITOR","HOLD","REJECT"], right=False).astype(str)


class AnomalyPipeline:
    def __init__(self, config=None):
        self.config = config or PipelineConfig()
        self.imputer = LotMedianImputer(self.config)
        self.lot_scorer = LotRelativeScorer(self.config)
        self.mv_scorer = MultivariateAnomalyScorer(self.config)
        self.drift_scorer = DriftSeverityScorer(self.config)
        self.absolute_layer = AbsoluteSpecLayer(self.config)

    def fit(self, df):
        validate_schema(df, self.config)
        df = self.imputer.fit_transform(df); df = add_drift_features(df, self.config)
        self.lot_scorer.fit(df); self.mv_scorer.fit(df); self.drift_scorer.fit(df)
        self.absolute_layer.fit(df)  # learns per-param datasheet limits, for live/new-data scoring
        return self

    def predict(self, df):
        validate_schema(df, self.config)
        original_index = df.index
        df = self.imputer.transform(df); df = add_drift_features(df, self.config)
        absolute_fail = self.absolute_layer.transform(df)
        lot_score, worst_z = self.lot_scorer.transform(df)
        mv_score = self.mv_scorer.transform(df)
        drift_score = self.drift_scorer.transform(df)
        risk_score = combine(lot_score, mv_score, drift_score, absolute_fail, self.config)
        decision = decide(risk_score, self.config)
        result = pd.DataFrame({
            self.config.id_col: df[self.config.id_col], self.config.lot_col: df[self.config.lot_col],
            self.config.group_col: df[self.config.group_col], "Absolute_Spec_Fail": absolute_fail,
            "Lot_Relative_Score": lot_score.round(1), "Worst_Lot_Zscore": worst_z.round(2),
            "Multivariate_Score": mv_score.round(1), "Drift_Score": drift_score.round(1),
            "Anomaly_Risk_Score": risk_score, "QA_Decision": decision,
        }, index=original_index)
        # Always emit the same columns whether or not ground truth exists yet, so downstream
        # code (frontend/backend) never has to branch on which columns are present.
        result[self.config.label_col] = df[self.config.label_col] if self.config.label_col in df.columns else pd.NA
        result[self.config.absolute_result_col] = df[self.config.absolute_result_col] if self.config.absolute_result_col in df.columns else pd.NA
        result['Absolute_Check_Source'] = 'ground_truth' if self.config.absolute_result_col in df.columns else 'proxy_live'
        return result

## Fit on `train` only


In [6]:
df = raw_df.merge(split_df[['Component_ID','split']], on='Component_ID')
df_train = df[df['split']=='train'].reset_index(drop=True)
df_val   = df[df['split']=='val'].reset_index(drop=True)
df_test  = df[df['split']=='test'].reset_index(drop=True)

base_config = PipelineConfig()
pipeline = AnomalyPipeline(base_config)
pipeline.fit(df_train)
print(f'fit on {len(df_train)} train rows')

fit on 5809 train rows


## Weights and thresholds are chosen to hit a target recall, not just AUC


In [7]:
def raw_layer_scores(fitted_pipeline, cfg, df_split):
    dfx = fitted_pipeline.imputer.transform(df_split)
    dfx = add_drift_features(dfx, cfg)
    absolute_fail = fitted_pipeline.absolute_layer.transform(dfx)
    lot_score, worst_z = fitted_pipeline.lot_scorer.transform(dfx)
    mv_score = fitted_pipeline.mv_scorer.transform(dfx)
    drift_score = fitted_pipeline.drift_scorer.transform(dfx)
    return lot_score, mv_score, drift_score, absolute_fail

lot_v, mv_v, drift_v, absfail_v = raw_layer_scores(pipeline, base_config, df_val)
label_v = df_val['Label'].values

TARGET_RECALL = 0.98

def threshold_for_recall(combined, label, target_recall):
    fail_mask = (label == 'Fail')
    n_fail = int(fail_mask.sum())
    if n_fail == 0:
        return None
    order = np.argsort(-combined)  # most suspicious first
    caught = 0
    for idx in order:
        if fail_mask[idx]:
            caught += 1
        if caught / n_fail >= target_recall:
            thr = combined[idx]
            flagged = combined >= thr
            recall = float((flagged & fail_mask).sum()) / n_fail
            safe_mask = (label == 'Safe')
            fa = float((flagged & safe_mask).sum()) / max(int(safe_mask.sum()), 1)
            return float(thr), recall, fa
    return None

rng = np.random.RandomState(0)
best = None  # (false_alarm_rate, -auc, weights, threshold_hold, achieved_recall, auc)
for _ in range(400):
    w = rng.dirichlet([1,1,1])
    combined = w[0]*lot_v + w[1]*mv_v + w[2]*drift_v
    combined = np.where(absfail_v==1, 100.0, combined)
    result = threshold_for_recall(combined, label_v, TARGET_RECALL)
    if result is None:
        continue
    thr_hold, recall, fa = result
    auc = roc_auc_score((label_v=='Fail').astype(int), combined)
    key = (fa, -auc)
    if best is None or key < best[0]:
        best = (key, w, thr_hold, recall, fa, auc)

if best is None:
    raise RuntimeError(
        f'No weight combination reached {TARGET_RECALL:.0%} recall on val Fail cases with the '
        'current features - lower TARGET_RECALL or add features/data before shipping this tier.'
    )

_, best_w, thr_hold, achieved_recall, achieved_fa, best_auc = best
print(f'selected weights (lot, multivariate, drift): {best_w.round(3)}')
print(f'-> {achieved_recall:.1%} recall on val Fail via HOLD/REJECT, at {achieved_fa:.1%} false-alarm rate on val Safe '
      f'(AUC={best_auc:.4f}, reported for context only)')

combined_v = best_w[0]*lot_v + best_w[1]*mv_v + best_w[2]*drift_v
combined_v = np.where(absfail_v==1, 100.0, combined_v)
safe_scores = combined_v[label_v=='Safe']

thr_monitor = min(float(np.percentile(safe_scores, 85)), thr_hold - 0.1)
thr_reject = max(float(np.percentile(safe_scores, 99.5)), thr_hold + 0.1)

tuned_config = PipelineConfig(
    weight_lot_relative=round(float(best_w[0]),3),
    weight_multivariate=round(float(best_w[1]),3),
    weight_drift=round(1 - round(float(best_w[0]),3) - round(float(best_w[1]),3), 3),
    threshold_monitor=round(thr_monitor,1), threshold_hold=round(thr_hold,1), threshold_reject=round(thr_reject,1),
)
print(tuned_config)

selected weights (lot, multivariate, drift): [0.095 0.235 0.67 ]
-> 98.0% recall on val Fail via HOLD/REJECT, at 13.3% false-alarm rate on val Safe (AUC=0.9544, reported for context only)
PipelineConfig(id_col='Component_ID', lot_col='Batch_ID', group_col='Part_Type', absolute_result_col='Traditional_Test_Result', absolute_fail_value='Fail', label_col='Label', params=['Leakage'], early_timepoints=['0h', '24h'], weight_lot_relative=0.095, weight_multivariate=0.235, weight_drift=0.67, threshold_monitor=15.5, threshold_hold=15.9, threshold_reject=96.8, z_score_cap=8.0, iforest_n_estimators=300, iforest_contamination='auto', iforest_min_group_size=30, random_state=42)


## Report train / val / test separately - test is the honest number


In [8]:
final_pipeline = AnomalyPipeline(tuned_config)
final_pipeline.fit(df_train)

train_results = final_pipeline.predict(df_train); train_results['split']='train'
val_results   = final_pipeline.predict(df_val);   val_results['split']='val'
test_results  = final_pipeline.predict(df_test);  test_results['split']='test'
all_results = pd.concat([train_results, val_results, test_results], ignore_index=True)

def generate_report(results, name):
    print(f'--- {name} (n={len(results)}) ---')
    counts = results['QA_Decision'].value_counts().reindex(['PASS','MONITOR','HOLD','REJECT']).fillna(0).astype(int)
    for k,v in counts.items():
        print(f'  {k:<10} {v:>6,} ({v/len(results):.1%})')
    fail_mask = results['Label']=='Fail'
    caught = results.loc[fail_mask,'QA_Decision'].isin(['HOLD','REJECT']).mean() if fail_mask.any() else float('nan')
    flag = '' if not fail_mask.any() or caught >= TARGET_RECALL else f'  <-- BELOW TARGET_RECALL ({TARGET_RECALL:.0%})'
    print(f'  Recall on true FAIL (HOLD/REJECT): {caught:.1%}{flag}')
    safe_mask = results['Label']=='Safe'
    false_alarm = results.loc[safe_mask,'QA_Decision'].isin(['HOLD','REJECT']).mean()
    print(f'  False-alarm rate on true SAFE (HOLD/REJECT): {false_alarm:.1%}')
    print()

generate_report(train_results, 'TRAIN (in-sample, optimistic - reference only')
generate_report(val_results,   'VAL (used to tune weights/thresholds - recall was tuned to hit TARGET_RECALL here)')
generate_report(test_results,  'TEST (held-out - the number to actually trust)')

--- TRAIN (in-sample, optimistic - reference only (n=5809) ---
  PASS        3,619 (62.3%)
  MONITOR        64 (1.1%)
  HOLD        1,639 (28.2%)
  REJECT        487 (8.4%)
  Recall on true FAIL (HOLD/REJECT): 94.1%  <-- BELOW TARGET_RECALL (98%)
  False-alarm rate on true SAFE (HOLD/REJECT): 13.5%

--- VAL (used to tune weights/thresholds - recall was tuned to hit TARGET_RECALL here) (n=1955) ---
  PASS        1,142 (58.4%)
  MONITOR        28 (1.4%)
  HOLD          632 (32.3%)
  REJECT        153 (7.8%)
  Recall on true FAIL (HOLD/REJECT): 98.0%
  False-alarm rate on true SAFE (HOLD/REJECT): 13.5%

--- TEST (held-out - the number to actually trust) (n=2236) ---
  PASS        1,420 (63.5%)
  MONITOR        22 (1.0%)
  HOLD          577 (25.8%)
  REJECT        217 (9.7%)
  Recall on true FAIL (HOLD/REJECT): 96.5%  <-- BELOW TARGET_RECALL (98%)
  False-alarm rate on true SAFE (HOLD/REJECT): 13.3%



## SHAP explanations for flagged components

In [9]:
flagged_test_mask = test_results['QA_Decision'].isin(['HOLD', 'REJECT'])
flagged_test_ids = set(test_results.loc[flagged_test_mask, 'Component_ID'])

# Need the (imputed, drift-featurized) feature frame, not the raw input, to feed the scorer
dfx_test = final_pipeline.imputer.transform(df_test)
dfx_test = add_drift_features(dfx_test, tuned_config)
dfx_test_flagged = dfx_test[dfx_test['Component_ID'].isin(flagged_test_ids)]

shap_explanations = final_pipeline.mv_scorer.shap_top_features(dfx_test_flagged, top_k=3)
shap_by_id = dict(zip(dfx_test_flagged['Component_ID'], shap_explanations.values))

all_results['SHAP_Top_Anomaly_Features'] = all_results['Component_ID'].map(shap_by_id)
print(f'SHAP attributions computed for {len(shap_explanations)} flagged TEST components')
all_results.loc[all_results['split']=='test', ['Component_ID','QA_Decision','SHAP_Top_Anomaly_Features']].dropna(subset=['SHAP_Top_Anomaly_Features']).head(10)

ExactExplainer explainer: 295it [00:29,  8.31it/s]                         
ExactExplainer explainer: 404it [00:31,  9.12it/s]                         

SHAP attributions computed for 794 flagged TEST components


,Component_ID,QA_Decision,SHAP_Top_Anomaly_Features
7769,C520,HOLD,"Leakage_0h(+0.03), Leakage_delta_24h(-0.02), L..."
7773,C530,HOLD,"Leakage_24h(-0.01), Leakage_0h(-0.01), Leakage..."
7783,C554,HOLD,"Leakage_delta_24h(+0.04), Leakage_pct_change_2..."
7785,C558,HOLD,"Leakage_pct_change_24h(+0.08), Leakage_delta_2..."
7787,C569,HOLD,"Leakage_pct_change_24h(+0.03), Leakage_0h(+0.0..."
7789,C575,REJECT,"Leakage_0h(+0.01), Leakage_pct_change_24h(-0.0..."
7790,C580,HOLD,"Leakage_delta_24h(+0.02), Leakage_24h(+0.02), ..."
7791,C586,REJECT,"Leakage_pct_change_24h(+0.05), Leakage_delta_2..."
7797,C603,REJECT,"Leakage_delta_24h(+0.05), Leakage_24h(+0.03), ..."
7799,C611,HOLD,"Leakage_delta_24h(+0.03), Leakage_24h(+0.01), ..."


## Reuse on new data

In [10]:
# Simulate what a REAL new component looks like: only 0h/24h has happened yet, no 96h/168h,
# no Traditional_Test_Result (the static check isn't final yet) and no Label (nobody knows the
# outcome - that's the point). Drop those columns entirely, not just blank them out, to prove
# the pipeline doesn't secretly depend on them being present.
new_batch_raw = df_test.sample(20, random_state=7).drop(
    columns=[c for c in df_test.columns if c.endswith('_96h') or c.endswith('_168h')]
            + ['Traditional_Test_Result', 'Label', 'split'],
    errors='ignore',
)
new_batch_scored = final_pipeline.predict(new_batch_raw)
new_batch_scored.head(20)


,Component_ID,Batch_ID,Part_Type,Absolute_Spec_Fail,Lot_Relative_Score,Worst_Lot_Zscore,Multivariate_Score,Drift_Score,Anomaly_Risk_Score,QA_Decision,Label,Traditional_Test_Result,Absolute_Check_Source
2195,C10307,ISRO-SCL-250217-07,Voltage Regulator IC,0,98.6,7.89,76.9,46.6,58.7,HOLD,<NA>,<NA>,proxy_live
51,C686,ISRO-SCL-250217-07,Voltage Regulator IC,0,9.3,0.75,3.1,7.6,6.7,PASS,<NA>,<NA>,proxy_live
747,C3749,ISRO-SCL-250217-07,Voltage Regulator IC,0,100.0,11.36,97.2,61.6,73.6,HOLD,<NA>,<NA>,proxy_live
34,C605,ISRO-SCL-250127-04,Digital Logic IC,0,8.0,0.64,5.8,6.7,6.6,PASS,<NA>,<NA>,proxy_live
2084,C9826,ISRO-SCL-250217-07,Voltage Regulator IC,0,5.3,0.43,0.7,7.8,5.9,PASS,<NA>,<NA>,proxy_live
412,C2205,ISRO-SCL-250203-05,MOSFET,0,12.7,1.02,14.0,6.1,8.6,PASS,<NA>,<NA>,proxy_live
1168,C5668,ISRO-SCL-250303-09,Voltage Regulator IC,0,15.8,1.26,44.0,15.9,22.5,HOLD,<NA>,<NA>,proxy_live
927,C4604,ISRO-SCL-250217-07,Voltage Regulator IC,0,1.9,0.16,2.2,9.1,6.8,PASS,<NA>,<NA>,proxy_live
1085,C5270,ISRO-SCL-250217-07,Voltage Regulator IC,0,23.3,1.86,13.9,19.3,18.4,HOLD,<NA>,<NA>,proxy_live
60,C719,ISRO-SCL-250203-05,MOSFET,0,7.2,0.57,3.4,9.0,7.5,PASS,<NA>,<NA>,proxy_live


## Save

In [11]:
all_results.to_csv('anomaly_results.csv', index=False)
joblib.dump(final_pipeline, 'anomaly_pipeline.joblib')

anomaly_shap_bundle = {
    'iforest_models': final_pipeline.mv_scorer.models_,           # dict[str, sklearn Pipeline] - safe
    'score_bounds': final_pipeline.mv_scorer.score_bounds_,       # dict[str, (p1, p99)]
    'background': final_pipeline.mv_scorer.background_,           # dict[str, pd.DataFrame]
    'feature_cols': final_pipeline.mv_scorer.feature_cols,        # list[str]
    'group_col': tuned_config.group_col,                          # 'Part_Type'
    'lot_col': tuned_config.lot_col,                              # 'Batch_ID'
    'params': tuned_config.params,                                # ['Leakage']
    'early_timepoints': tuned_config.early_timepoints,            # ['0h','24h']
    'early_value_cols': tuned_config.early_value_cols(),          # for replicating imputation
    'group_medians': final_pipeline.imputer.group_medians_,       # dict[col][lot] -> median
    'global_medians': final_pipeline.imputer.global_medians_,     # dict[col] -> median
}
joblib.dump(anomaly_shap_bundle, 'anomaly_shap_bundle.joblib')
print('saved anomaly_results.csv, anomaly_pipeline.joblib (session-local), and anomaly_shap_bundle.joblib (portable, for Module C)')

saved anomaly_results.csv, anomaly_pipeline.joblib (session-local), and anomaly_shap_bundle.joblib (portable, for Module C)
